# **Compulsory Assignment 2**
## *Machine Learning and Deep Learning (CDSCO2041C)*
*Group: MLS26_CA02*

*Student IDs: 185912, 161989, 160714 & 160363*

*Dataset: _eth_dkk.csv & _XCSE_NOVO-1.csv*

---

# Investment Choice between Ethereum and Novo Nordisk using RoI
## Introduction

In this report, our group investigates how an investor with DKK 250,000 in available cash should allocate capital between a cryptocurrency and an individual equity. Following the assignment setup, we focus on Ethereum as the cryptocurrency candidate, motivated by its lower price level relative to Bitcoin and its perceived growth potential, while Novo Nordisk represents the equity alternative due to the availability of historical market data for one of Denmark’s largest listed companies.


Our analysis is structured around Return on Investment (RoI) as the main performance metric, using the provided datasets for Ethereum priced in DKK and for Novo Nordisk listed on the Copenhagen exchange. We first establish a consistent data foundation by cleaning, aligning, and transforming the two time series into comparable return and RoI measures. Building on this, we evaluate which asset has historically offered the highest risk adjusted returns and whether the observed differences are stable across market conditions.


---

## 1. Asset Selection

Should you invest in cryptocurrency (Ethereum) or in equity (Novo Nordisk stock)?

---

In [1]:
# -----------------------------
# Imports
# -----------------------------
import os
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

# Reproducibility
np.random.seed(42)


In [2]:
# -----------------------------
# Style
# -----------------------------

def set_clean_theme():
    """Apply consistent seaborn and matplotlib styling."""
    sns.set_theme(style="white", context="notebook")
    plt.rcParams["figure.facecolor"] = "white"
    plt.rcParams["axes.facecolor"] = "white"
    plt.rcParams["axes.grid"] = False

def clean_ax(ax):
    """Apply minimal axis styling."""
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", labelsize=9)

def caption_title(title: str) -> str:
    """Lightweight caption formatter used for figure/table titles."""
    if title is None:
        return ""
    t = str(title).strip()
    return t[:-1] if t.endswith(".") else t

# Standard figure geometry (consistent width)
FIG_W = 9.5

def fig_ax(height: float):
    """Create a standard figure with consistent width and sizing."""
    fig, ax = plt.subplots(figsize=(FIG_W, height))
    return fig, ax

# Report numbering helpers
TABLE_START = 1
FIGURE_START = 1
table_no = TABLE_START
figure_no = FIGURE_START

def print_table(title):
    global table_no
    print(f"Table {table_no}: {caption_title(title)}")
    table_no += 1

def print_figure(title):
    global figure_no
    print(f"Figure {figure_no}: {caption_title(title)}")
    figure_no += 1

# Palette choice (keep consistent, minimal, readable)
ASSET_LABELS = {"eth": "Ethereum (DKK)", "novo": "Novo Nordisk (XCSE)"}
PAL_10 = sns.color_palette("rocket_r", 10)
ASSET_PALETTE = {
    ASSET_LABELS["eth"]: PAL_10[2],
    ASSET_LABELS["novo"]: PAL_10[6],
}

set_clean_theme()

In [3]:
# Load, clean, sort, and store ETH and NOVO as full dataframes, plus merged on Novo dates
ETH_Path = "Data/_eth_dkk.csv"
NOVO_Path = "Data/_XCSE_NOVO-1.csv"

eth_raw = pd.read_csv(ETH_Path, sep=";", engine="python")
novo_raw = pd.read_csv(NOVO_Path, sep=",", engine="python", skipinitialspace=True, quotechar='"')

eth_raw.columns = eth_raw.columns.str.replace('"', "", regex=False).str.strip()
novo_raw.columns = novo_raw.columns.str.replace('"', "", regex=False).str.strip()

eth_raw = eth_raw.rename(columns={
    "Dato": "Date",
    "Åbning": "Open",
    "Høj": "High",
    "Lav": "Low",
    "Luk": "Close",
    "Volumen": "Volume",
    "Markedsværdi": "Market Cap",
    "Ændring, %": "Change, %"
})

eth_raw["Date"] = pd.to_datetime(
    eth_raw["Date"].astype(str).str.replace('"', "", regex=False).str.strip(),
    format="%Y-%m-%d",
    errors="coerce"
)

novo_raw["Date"] = pd.to_datetime(
    novo_raw["Date"].astype(str).str.replace('"', "", regex=False).str.strip(),
    format="%m/%d/%Y",
    errors="coerce"
)

for col in ["Open", "High", "Low", "Close", "Volume"]:
    if col in eth_raw.columns:
        eth_raw[col] = eth_raw[col].astype(str).str.replace('"', "", regex=False).str.strip()
        eth_raw[col] = eth_raw[col].str.replace(",", "", regex=False)
        eth_raw[col] = pd.to_numeric(eth_raw[col], errors="coerce")
    if col in novo_raw.columns:
        novo_raw[col] = novo_raw[col].astype(str).str.replace('"', "", regex=False).str.strip()
        novo_raw[col] = novo_raw[col].str.replace(",", "", regex=False)
        novo_raw[col] = novo_raw[col].str.replace(" ", "", regex=False)
        novo_raw[col] = novo_raw[col].str.replace(r"^\.", "", regex=True)
        novo_raw[col] = pd.to_numeric(novo_raw[col], errors="coerce")

eth_df = eth_raw.dropna(subset=["Date"]).drop_duplicates("Date").sort_values("Date").reset_index(drop=True)
novo_df = novo_raw.dropna(subset=["Date"]).drop_duplicates("Date").sort_values("Date").reset_index(drop=True)

eth_close = eth_df[["Date", "Close"]].dropna().rename(columns={"Date": "date", "Close": "close_eth"})
novo_close = novo_df[["Date", "Close"]].dropna().rename(columns={"Date": "date", "Close": "close_novo"})

merged_df = (
    pd.merge(eth_close, novo_close, on="date", how="inner")
      .sort_values("date")
      .reset_index(drop=True)
)

print_table("Cleaned Ethereum dataset (full columns, sorted by date)")
display(eth_df.head(3))

print_table("Cleaned Novo Nordisk dataset (full columns, sorted by date)")
display(novo_df.head(3))

print_table("Merged closing prices on common dates (Novo trading days)")
display(merged_df.head(3))

print("ETH rows (full):", len(eth_df))
print("NOVO rows (full):", len(novo_df))
print("Merged rows (Novo trading days):", len(merged_df))

Table 1: Cleaned Ethereum dataset (full columns, sorted by date)


,Date,Open,High,Low,Close,Volume,Market Cap,"Change, %"
0,2025-03-01,16081.11,16394.10,15399.37,15935.94,NaN,1.92T,-0.99
1,2025-03-02,15936.01,18330.51,15610.02,18097.15,NaN,2.18T,13.56
2,2025-03-03,17932.04,17970.78,14939.64,15303.53,NaN,1.85T,-15.44


Table 2: Cleaned Novo Nordisk dataset (full columns, sorted by date)


,Date,Open,High,Low,Close,Volume
0,2025-02-27,630.0,645.8,628.3,641.8,2974061
1,2025-02-28,641.0,648.2,631.2,644.5,7285161
2,2025-03-03,650.0,651.2,636.2,639.1,2242871


Table 3: Merged closing prices on common dates (Novo trading days)


,date,close_eth,close_novo
0,2025-03-03,15303.53,639.1
1,2025-03-04,15280.70,616.2
2,2025-03-05,15494.54,631.5


ETH rows (full): 366
NOVO rows (full): 250
Merged rows (Novo trading days): 248


In [5]:
# Create standalone price series and aligned series on Novo trading days
eth_px = eth_df[["Date", "Close"]].dropna().rename(columns={"Date": "date", "Close": "close_eth"}).copy()
novo_px = novo_df[["Date", "Close"]].dropna().rename(columns={"Date": "date", "Close": "close_novo"}).copy()

eth_px = eth_px.sort_values("date").reset_index(drop=True)
novo_px = novo_px.sort_values("date").reset_index(drop=True)

aligned_px = merged_df.copy()
aligned_px = aligned_px.sort_values("date").reset_index(drop=True)

display(eth_px.head(3))
display(novo_px.head(3))
display(aligned_px.head(3))

,date,close_eth
0,2025-03-01,15935.94
1,2025-03-02,18097.15
2,2025-03-03,15303.53


,date,close_novo
0,2025-02-27,641.8
1,2025-02-28,644.5
2,2025-03-03,639.1


,date,close_eth,close_novo
0,2025-03-03,15303.53,639.1
1,2025-03-04,15280.70,616.2
2,2025-03-05,15494.54,631.5


In [7]:
# Compute log returns, simple returns, and cumulative RoI for each asset on its own calendar
def add_returns_and_roi(df, price_col):
    out = df.copy()
    out["log_price"] = np.log(out[price_col])
    out["log_ret"] = out["log_price"].diff()
    out["ret"] = out[price_col].pct_change()
    out["roi"] = out[price_col] / out[price_col].iloc[0] - 1.0
    return out

eth_ts = add_returns_and_roi(eth_px, "close_eth")
novo_ts = add_returns_and_roi(novo_px, "close_novo")

display(eth_ts.head(5))
display(novo_ts.head(5))

,date,close_eth,log_price,log_ret,ret,roi
0,2025-03-01,15935.94,9.676332,NaN,NaN,0.000000
1,2025-03-02,18097.15,9.803510,0.127178,0.135619,0.135619
2,2025-03-03,15303.53,9.635839,-0.167671,-0.154368,-0.039685
3,2025-03-04,15280.70,9.634346,-0.001493,-0.001492,-0.041117
4,2025-03-05,15494.54,9.648243,0.013897,0.013994,-0.027698


,date,close_novo,log_price,log_ret,ret,roi
0,2025-02-27,641.8,6.464277,NaN,NaN,0.000000
1,2025-02-28,644.5,6.468475,0.004198,0.004207,0.004207
2,2025-03-03,639.1,6.460061,-0.008414,-0.008379,-0.004207
3,2025-03-04,616.2,6.423572,-0.036489,-0.035832,-0.039888
4,2025-03-05,631.5,6.448098,0.024526,0.024830,-0.016049


In [8]:
# Compute returns and RoI on the aligned dataset to support comparisons and portfolio work
aligned_ts = aligned_px.copy()
aligned_ts["log_ret_eth"] = np.log(aligned_ts["close_eth"]).diff()
aligned_ts["log_ret_novo"] = np.log(aligned_ts["close_novo"]).diff()
aligned_ts["ret_eth"] = aligned_ts["close_eth"].pct_change()
aligned_ts["ret_novo"] = aligned_ts["close_novo"].pct_change()
aligned_ts["roi_eth"] = aligned_ts["close_eth"] / aligned_ts["close_eth"].iloc[0] - 1.0
aligned_ts["roi_novo"] = aligned_ts["close_novo"] / aligned_ts["close_novo"].iloc[0] - 1.0

display(aligned_ts.head(3))

,date,close_eth,close_novo,log_ret_eth,log_ret_novo,ret_eth,ret_novo,roi_eth,roi_novo
0,2025-03-03,15303.53,639.1,NaN,NaN,NaN,NaN,0.000000,0.000000
1,2025-03-04,15280.70,616.2,-0.001493,-0.036489,-0.001492,-0.035832,-0.001492,-0.035832
2,2025-03-05,15494.54,631.5,0.013897,0.024526,0.013994,0.024830,0.012481,-0.011892


---
# 2. Capital Allocation Strategy 

How should the DKK 250,000 be allocated? Should you adopt a pure 
strategy (100% allocation to one asset) or a hybrid strategy (e.g., X% in cryptocurrency and Y% in stock)?
What is the appropriate investment horizon (short-term vs. long-term; months vs. years)?

---

Next, we translate the asset level results into a portfolio decision. We examine whether a pure strategy, allocating 100% of the capital to either Ethereum or Novo Nordisk, is justified, or whether diversification benefits support a hybrid allocation across both assets. This section also addresses investment horizon by comparing outcomes over shorter horizons measured in months versus longer horizons measured in years, with attention to how volatility and drawdowns affect the practical feasibility of each strategy. 

---
# 3. Market Timing and Seasonality 

Are there identifiable seasonal patterns or cyclical trends in cryp-tocurrency and stock returns?
For example, are there specific months or periods that historically
yield higher average returns, suggesting a more favorable entry point?

---

Finally, we explore market timing and seasonality. Specifically, we test whether returns exhibit recurring monthly or cyclical patterns that could indicate more favorable entry periods, and we complement this with a regime based perspective that distinguishes between low volatility and high volatility phases. Where relevant, we additionally highlight extreme events and crash like periods to understand tail risk and its implications for allocation decisions.